In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
import re

In [2]:
folder = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"

### Warnings from Network Check
- ran with Zones with demand data

In [3]:
warnings_txt = os.path.join(folder, "Zones Connectivity Test", "with final filt", "all_zones_more_than_1conn.txt")

with open(warnings_txt, "r", encoding="utf-8") as archivo:
    messages = archivo.read()

pairs = re.findall(
    r"between Zone (\d+) and Zone (\d+)",
    messages
)

errors = pd.DataFrame(
    pairs,
    columns=["origin_zone", "destination_zone"]
).astype(int)

print(f"Check network consistency: {len(errors):,} warnings")

errors

Check network consistency: 162,038 warnings


,origin_zone,destination_zone
0,1,443
1,1,1012
2,1,5028
3,1,5082
4,1,5092
...,...,...
162033,9053,8333
162034,9053,8373
162035,9053,8410
162036,9053,8413


In [4]:
origin_problems = (
    errors["origin_zone"]
    .value_counts()
    .rename_axis("zone_id")
    .reset_index(name="fails_as_origin")
)

destination_problems = (
    errors["destination_zone"]
    .value_counts()
    .rename_axis("zone_id")
    .reset_index(name="fails_as_destination")
)

zone_problems = (
    origin_problems
    .merge(
        destination_problems,
        on="zone_id",
        how="outer"
    )
    .fillna(0)
)

zone_problems[["fails_as_origin", "fails_as_destination"]] = (
    zone_problems[["fails_as_origin", "fails_as_destination"]].astype(int)
)


zone_problems["total"] = (
    zone_problems["fails_as_origin"]
    + zone_problems["fails_as_destination"]
)

zone_problems = zone_problems.sort_values(
    "total",
    ascending=False
)

zone_problems

,zone_id,fails_as_origin,fails_as_destination,total
454,1012,4119,4119,8238
812,5092,4119,4119,8238
1954,8373,4117,4117,8234
2064,8483,2095,2095,4190
2006,8425,2095,2095,4190
...,...,...,...,...
688,4069,19,1,20
650,4016,11,2,13
637,4003,11,2,13
707,4094,11,1,12


In [5]:
# 2. Cuántas zonas tienen cada cantidad de fallas como origen
print(zone_problems["fails_as_origin"].value_counts().sort_index())
print()
# 3. Cuántas zonas tienen cada cantidad de fallas como destino
print(zone_problems["fails_as_destination"].value_counts().sort_index())

fails_as_origin
10        11
11        12
19         1
20        60
21         1
23         2
24        60
33         6
34         5
35         1
36         7
37         2
38      1988
83         1
84         1
2094       2
2095       6
4115       5
4117       6
4119       5
Name: count, dtype: int64

fails_as_destination
1          4
2         62
23         1
24        60
36         6
38        16
39         2
40      1987
74         9
75        10
84         1
2089       1
2093       1
2094       1
2095       7
4108       2
4111       2
4117       2
4119       8
Name: count, dtype: int64


In [6]:
zone_problems[
    (zone_problems["fails_as_origin"] > 1000)
    | (zone_problems["fails_as_destination"] > 1000)
]

,zone_id,fails_as_origin,fails_as_destination,total
454,1012,4119,4119,8238
812,5092,4119,4119,8238
1954,8373,4117,4117,8234
2064,8483,2095,2095,4190
2006,8425,2095,2095,4190
1536,7188,2095,2095,4190
2181,9054,2095,2095,4190
748,5028,2095,2095,4190
938,5218,2095,2095,4190
2021,8440,2094,2094,4188


In [20]:
zone_problems.to_csv(
    os.path.join(folder, "Zones Connectivity Test", "unreachable_zones_summary.csv"),
    index=False
)

In [31]:
isolated_group = zone_problems[
    zone_problems["fails_as_origin"] == 29
].copy()

print(len(isolated_group))
isolated_group["zone_id"].tolist()

2043


[2,
 3,
 4,
 5,
 6,
 7,
 8,
 80,
 65,
 66,
 67,
 68,
 69,
 70,
 13,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 9005,
 9006,
 8537,
 8538,
 8539,
 8540,
 8541,
 8542,
 9013,
 9014,
 8545,
 8546,
 9001,
 9002,
 9003,
 10,
 112,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 56,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 64,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 72,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 144,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 88,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 96,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 104,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 176,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 120,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 128,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 136,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 208,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 152,
 201,
 202,
 203,
 204,
 205,
 206,
 207,
 160,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 168,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 240,
 225,
 226,
 227,
 228,
 229,
 230,
 231,
 184,
 

In [8]:
zone_problems["fails_as_origin"].value_counts().sort_index()

fails_as_origin
43         1
139     2049
140        5
141        2
142        1
143        2
144        2
145        1
152        1
2106      96
2202      43
Name: count, dtype: int64

In [9]:
zone_problems["fails_as_destination"].value_counts().sort_index()

fails_as_destination
43         1
139     2015
140       47
141        1
2106      96
2202      43
Name: count, dtype: int64

In [10]:
# Zonas que prácticamente NO pueden salir
bad_origins = zone_problems[
    zone_problems["fails_as_origin"] > 2000
].sort_values("fails_as_origin", ascending=False)

# Zonas a las que prácticamente NO se puede llegar
bad_destinations = zone_problems[
    zone_problems["fails_as_destination"] > 2000
].sort_values("fails_as_destination", ascending=False)

# Zonas graves en ambos sentidos
bad_both = zone_problems[
    (zone_problems["fails_as_origin"] > 2000) &
    (zone_problems["fails_as_destination"] > 2000)
]

print("Bad origins:", len(bad_origins))
print("Bad destinations:", len(bad_destinations))
print("Bad both:", len(bad_both))

Bad origins: 139
Bad destinations: 139
Bad both: 139


In [11]:
bad_both[[
    "zone_id",
    "fails_as_origin",
    "fails_as_destination"
]].sort_values(
    "fails_as_origin",
    ascending=False
)

,zone_id,fails_as_origin,fails_as_destination
917,5176,2202,2202
686,4052,2202,2202
1016,5275,2202,2202
900,5159,2202,2202
959,5218,2202,2202
...,...,...,...
670,4036,2106,2106
719,4085,2106,2106
669,4035,2106,2106
721,4087,2106,2106


In [12]:
bad_both["fails_as_origin"].value_counts()

fails_as_origin
2106    96
2202    43
Name: count, dtype: int64

In [13]:
group_2202 = bad_both[
    bad_both["fails_as_origin"] == 2202
]

group_2106 = bad_both[
    bad_both["fails_as_origin"] == 2106
]

print("Completely disconnected:", len(group_2202))
print("Large isolated group:", len(group_2106))

Completely disconnected: 43
Large isolated group: 96


In [14]:
group_2202

,zone_id,fails_as_origin,fails_as_destination,total
917,5176,2202,2202,4404
686,4052,2202,2202,4404
1016,5275,2202,2202,4404
900,5159,2202,2202,4404
959,5218,2202,2202,4404
978,5237,2202,2202,4404
698,4064,2202,2202,4404
651,4017,2202,2202,4404
678,4044,2202,2202,4404
804,5063,2202,2202,4404
